In [1]:
import pyurg
import numpy as np
import matplotlib.pyplot as plt
from numpy import linalg as la
import time
from pprint import pprint
import json
from scipy.stats import linregress

# urg = pyurg.UrgDevice()
# if not urg.connect():
#     print('Could not connect.')

In [ ]:
datas = {}
for i in range(1,5):
    with open(f"measures/mesure_{i}.json") as f:
        datas[i-1] = np.array(json.load(f))

In [3]:
# datas, timestamp = urg.capture()

In [4]:
# angles = []
# for i in range(len(datas)):
#     angles.append(urg.index2rad(i))

In [5]:
#Filter points by distance, and by angle, returns np vector
def filter_points(dist, angle, verbose):
    max_distance = 1000
    min_distance = 60
    min_angle = np.pi/2
    max_angle = 3*np.pi/2
    if verbose:print(f"Filtering points by : (max_d:{max_distance}, min_d:{min_distance}, max_a:{max_angle}, min_a:{min_angle})")
    vector = np.array(list(zip(dist, angle)))
    vector = vector[np.logical_and(vector[:, 0] > min_distance, vector[:, 0] < max_distance)]  # Filter out data points with distance less than 1000
    vector = vector[np.logical_and(vector[:, 1] > min_angle, vector[:, 1] < max_angle)] #Filtrer la moitié des points
    return vector

#remove outliers points and transform polar values to cartesian, return np coords vector 
def remove_outliers_2cart(vector, verbose):
    x = vector[:, 0] * np.cos(vector[:, 1])
    y = vector[:, 0] * np.sin(vector[:, 1])
    dist_min = 6
    coords = []
    for i in range(0,len(vector[:,0])-2):
        p1 = np.array([x[i], y[i]])
        p2 = np.array([x[i+1], y[i+1]])
        p3 = np.array([x[i+2], y[i+2]])
        
        if la.norm(p1-p3) < dist_min:
            if la.norm(p2-p3) <= dist_min or la.norm(p1-p2) <= dist_min:
                coords.append([x[i], y[i]])
            else:
                if verbose:print(f"Pt abberant : ({x[i]};{y[i]})")
        else:
            coords.append([x[i], y[i]])
    return np.array(coords)

#plot points and center points
def plot_points(coords):
    plt.scatter(coords[:,0], coords[:,1], marker="+")
    plt.scatter(0,0)

#cluster points according to distance and alignment, returns cluster dict
def cluster_points(coords, corner_limit, verbose):
    d_max = 50
    clusters = {}
    curr_cluster = 0
    min_cluster_size = 50
    max_error_reg =30

    for i in range(0,len(coords[:,0])-1):
        if i==0:
            clusters[curr_cluster] = []
            clusters[curr_cluster].append(coords[i,:])

        elif la.norm(coords[i,:]-coords[i-1,:]) <= d_max:
            #look if distance between 2 points is under d_max
            
            if len(clusters[curr_cluster]) >= 3:
                reg = linregress(np.array(clusters[curr_cluster])[:,0], np.array(clusters[curr_cluster])[:,1])
                err = abs(reg.slope*coords[i,0] + reg.intercept - coords[i,1])
                if err < max_error_reg:
                    clusters[curr_cluster].append(coords[i,:])
                else:
                    if (len(clusters[curr_cluster]) > min_cluster_size):
                        curr_cluster += 1
                        if verbose:print(f"New cluster : {coords[i,:]}")
                    clusters[curr_cluster] = []
                    clusters[curr_cluster].append(coords[i,:])
            else:
                clusters[curr_cluster].append(coords[i,:])
        else:
            if (len(clusters[curr_cluster]) > min_cluster_size):
                curr_cluster += 1
            clusters[curr_cluster] = []
            clusters[curr_cluster].append(coords[i,:])

    # ----- Keeping only 2 main clusters
    if corner_limit:
        if len(clusters) > 2:
            if verbose: print(f"Too much clusters : {len(clusters)}")
            sorted_cluster_keys = sorted(clusters, key=lambda k: len(clusters[k]), reverse=True)
            clusters = {0: clusters[sorted_cluster_keys[0]], 1:clusters[sorted_cluster_keys[1]]}
    # -----
    return clusters

#Plot clusters
def plot_clusters(clusters):
    for cluster in clusters.values():
        cluster = np.array(cluster)
        plt.scatter(cluster[:,0], cluster[:,1], marker="+")

In [6]:
from sklearn.linear_model import RANSACRegressor, LinearRegression
 
def detect_line1(cluster):
    ransac = RANSACRegressor(
    #    residual_threshold=10,  # Très strict : 1cm de tolérance max
    # min_samples=2,            # On utilise le strict minimum pour définir une ligne
    # max_trials=1000,          # On lui donne plus de chances de réussir
        stop_probability=0.99     # On s'assure d'une haute confiance statistique
    )
    ransac.fit(cluster[:,0].reshape(-1, 1),cluster[:,1])
    return ransac.estimator_

def detect_line2(cluster, verbose):
    model = LinearRegression()
    model.fit(cluster[:,0].reshape(-1, 1),cluster[:,1])
    if (verbose):print(model.coef_)
    return model

def detect_intersection(line1_model, line2_model, verbose):
    a1, b1 = line1_model.coef_[0], line1_model.intercept_
    a2, b2 = line2_model.coef_[0], line2_model.intercept_

    x_corner = (b2 - b1) / (a1 - a2)
    y_corner = a1 * x_corner + b1
    if (verbose):print(f"Coin détecté en : ({x_corner:.3f}, {y_corner:.3f})")
    return np.array([x_corner, y_corner])

In [7]:
def plot_lidar_detection(points, line1_model, line2_model, corner):
    plt.figure()
    
    # 1. Afficher tous les points (en gris léger pour le fond)
    plt.scatter(points[:, 0], points[:, 1], c='lightgrey', label='Tous les points', marker="+")
    
    # 3. Tracer les lignes de régression
    # On définit une plage de X pour tracer la ligne
    x_range = np.linspace(points[:, 0].min(), points[:, 0].max(), 100).reshape(-1, 1)
    
    y1_plot = line1_model.predict(x_range)
    y2_plot = line2_model.predict(x_range)
    
    plt.plot(x_range, y1_plot, 'b--', alpha=0.6, label='Modèle Ligne 1')
    plt.plot(x_range, y2_plot, 'r--', alpha=0.6, label='Modèle Ligne 2')
    
    # 4. Afficher le coin (l'intersection)
    plt.scatter(corner[0], corner[1], c='yellow', edgecolors='black', 
                s=100, marker='*', label='Coin détecté', zorder=5)
    
    plt.xlabel('X (mètres)')
    plt.ylabel('Y (mètres)')
    plt.title('Détection de coin sur nuage de points LIDAR 2D')
    plt.legend()
    plt.axis('equal') # Important pour ne pas déformer la géométrie
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.show()

In [40]:
def detect_rotation(line1_model, line2_model):
    pente1 = line1_model.coef_[0]
    pente2 = line2_model.coef_[0]

    angle1 = np.degrees(np.arctan(pente1))
    angle2 = np.degrees(np.arctan(pente2))
    err = abs(angle1 - angle2) -90
    print(f"Erreur d'angle entre les deux droites : {err}")
    biss = (angle1 + angle2)/2
    theta = biss-45
    print(f"Correction : -{theta}")
    correction = - np.deg2rad(theta)

In [36]:
def compute_offset(data, verbose):
    angles = []
    for i in range(0,len(data)):
        angles.append(np.deg2rad((i*360)/len(data)))
    vector = filter_points(data, angles, verbose)
    coords = remove_outliers_2cart(vector, verbose)
    # plot_points(coords)
    clusters = cluster_points(coords, True, verbose)
    # plot_clusters(clusters)
    line1_model = detect_line2(np.array(clusters[0][:]), verbose)
    line2_model = detect_line2(np.array(clusters[1][:]), verbose)
    detect_rotation(line1_model, line2_model)

    intersection = detect_intersection(line1_model, line2_model, verbose)
    if verbose: plot_lidar_detection(coords, line1_model, line2_model, intersection)
    return intersection

In [41]:
for data in datas.values():
    point = compute_offset(data, False)
    print(point)

Erreur d'angle entre les deux droites : 21.515924310536462
Correction : --45.77338674068359
[-142.97189065    2.98065068]
Erreur d'angle entre les deux droites : 21.02145541683541
Correction : --46.43047831783629
[-133.48999386    4.07839235]
Erreur d'angle entre les deux droites : 28.598618786263813
Correction : --45.90155100256841
[-160.12143222    4.40251945]
Erreur d'angle entre les deux droites : 37.039368887956414
Correction : --50.6428304718097
[-227.91448256   30.56683487]


In [ ]:
x_nouveau = 